In [7]:
%pip install pandas

  Using cached pandas-3.1.1-cp312-cp312-win_amd64.whl.metadata (16 kB)
  Using cached numpy-2.5.5-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached pandas-3.1.1-cp312-cp312-win_amd64.whl (10.8 MB)
Using cached numpy-2.5.5-cp312-cp312-win_amd64.whl (12.6 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---

In [8]:
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import joblib

# CSV 파일 경로 설정
def get_csv_path(filename):
    data_path = Path("data") / filename
    return data_path if data_path.exists() else Path(filename)

# 더미데이터 불러오기
purchase_df = pd.read_csv(
    get_csv_path("dummy_purchase_history.csv"),
    encoding="utf-8-sig"
)

products_df = pd.read_csv(
    get_csv_path("dummy_products.csv"),
    encoding="utf-8-sig"
)

members_df = pd.read_csv(
    get_csv_path("dummy_members.csv"),
    encoding="utf-8-sig"
)

# 날짜 데이터 형식 변환
purchase_df["ordered_at"] = pd.to_datetime(
    purchase_df["ordered_at"]
)

print("구매 이력:", purchase_df.shape)
print("상품 정보:", products_df.shape)
print("회원 정보:", members_df.shape)

print("\n구매 이력 샘플")
display(purchase_df.head(3))

print("\n회원 정보 샘플")
display(members_df.head(3))


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
category_prices = {
    "CPU": 350000, "GPU": 650000, "MAINBOARD": 210000,
    "RAM": 130000, "SSD": 110000, "HDD": 100000,
    "POWER": 120000, "CASE": 80000, "KEYBOARD": 70000,
    "MOUSE": 40000, "SPEAKER": 90000, "MONITOR": 280000
}

purchase_df["unit_price"] = purchase_df["category"].map(category_prices)
if purchase_df["unit_price"].isna().any():
    raise ValueError("가격이 지정되지 않은 카테고리가 있습니다.")

purchase_df["item_amount"] = purchase_df["unit_price"] * purchase_df["quantity"]
print(purchase_df[["member_id", "order_id", "item_amount", "ordered_at"]].head())

In [ ]:
training_rows = []

for base_month in pd.period_range("2026-03", "2026-09", freq="M"):
    # 기준 월이 끝나는 시각 (= 다음 달 1일 0시)
    next_start = (base_month + 1).start_time
    next_end = (base_month + 2).start_time

    # ① 기준 월까지의 구매 이력만 선택
    history = purchase_df[purchase_df["ordered_at"] < next_start]

    # ② 회원별 RFM 집계 (기존 customer01에서 배운 groupby)
    group = history.groupby("member_id")
    customer = pd.DataFrame({
        "purchase_count": group["order_id"].nunique(),
        "total_spent": group["item_amount"].sum(),
        "last_purchase": group["ordered_at"].max()
    })
    customer["recency_days"] = (next_start - customer["last_purchase"]).dt.days

    # ③ 다음 달 구매 회원 = 학습할 정답
    next_buyers = set(
        purchase_df.loc[
            (purchase_df["ordered_at"] >= next_start)
            & (purchase_df["ordered_at"] < next_end),
            "member_id"
        ]
    )
    customer["bought_next_month"] = customer.index.isin(next_buyers).astype(int)
    customer["base_month"] = str(base_month)
    training_rows.append(customer.reset_index())

training_df = pd.concat(training_rows, ignore_index=True)
print(training_df[["member_id", "base_month", "recency_days", "purchase_count", "total_spent", "bought_next_month"]].head(10))
print("학습용 사례 총:", len(training_df), "건")
print("정답 비율:")
print(training_df["bought_next_month"].value_counts())

In [ ]:
FEATURES = ["recency_days", "purchase_count", "total_spent"]
TARGET = "bought_next_month"

train_df = training_df[training_df["base_month"] <= "2026-07"]
valid_df = training_df[training_df["base_month"] == "2026-08"]
test_df = training_df[training_df["base_month"] == "2026-09"]

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_valid, y_valid = valid_df[FEATURES], valid_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

print("학습:", len(X_train), "건 | 검증:", len(X_valid), "건 | 시험:", len(X_test), "건")
print("시험 정답 분포:", y_test.value_counts().to_dict())

In [ ]:
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf", C=1.0, gamma="scale",
        class_weight="balanced", probability=True, random_state=42
    ))
])

svm_model.fit(X_train, y_train)  # ★ 실제 머신러닝 학습
print("SVM 모델 학습 완료")

In [ ]:
valid_pred = svm_model.predict(X_valid)
valid_prob = svm_model.predict_proba(X_valid)[:, 1]

print("검증 정확도:", round(accuracy_score(y_valid, valid_pred), 4))
print("검증 정밀도:", round(precision_score(y_valid, valid_pred, zero_division=0), 4))
print("검증 재현율:", round(recall_score(y_valid, valid_pred, zero_division=0), 4))
print("검증 F1:", round(f1_score(y_valid, valid_pred, zero_division=0), 4))
print("검증 ROC-AUC:", round(roc_auc_score(y_valid, valid_prob), 4))
print("검증 혼동 행렬:\n", confusion_matrix(y_valid, valid_pred))
print("항상 미구매(0) 예측의 정확도:", round((y_valid == 0).mean(), 4))

In [ ]:
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf", C=1.0, gamma="scale",
        class_weight="balanced", probability=True, random_state=42
    ))
])

train_valid_df = training_df[training_df["base_month"] <= "2026-08"]
final_model.fit(train_valid_df[FEATURES], train_valid_df[TARGET])

test_pred = final_model.predict(X_test)
test_prob = final_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "accuracy": round(float(accuracy_score(y_test, test_pred)), 4),
    "precision": round(float(precision_score(y_test, test_pred, zero_division=0)), 4),
    "recall": round(float(recall_score(y_test, test_pred, zero_division=0)), 4),
    "f1": round(float(f1_score(y_test, test_pred, zero_division=0)), 4),
    "roc_auc": round(float(roc_auc_score(y_test, test_prob)), 4),
}
print("최종 시험 성능:", test_metrics)
print("혼동 행렬:\n", confusion_matrix(y_test, test_pred))
print("항상 미구매 예측의 정확도:", round(float((y_test == 0).mean()), 4))

In [ ]:
model_file = "customer_svm_model.pkl"
joblib.dump(final_model, model_file)
loaded_model = joblib.load(model_file)

# 실제 시험 데이터에서 첫 번째 고객의 시점별 특징만 이용해 예측
sample = X_test.iloc[[0]]
print("예측 값 (0=다음 달 미구매, 1=구매):", int(loaded_model.predict(sample)[0]))
print("해당 고객의 실제 다음 달 구매 여부:", int(y_test.iloc[0]))
print("모델 저장:", model_file)

training_df.to_csv("customer_svm_training_data.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([test_metrics]).to_csv("customer_svm_test_metrics.csv", index=False, encoding="utf-8-sig")
print("학습 데이터와 최종 평가 CSV도 저장했습니다.")